## 실시간 비디오 증강 앱 프로젝트

웹캠 영상에 실시간 필터와 밝기 조절을 적용하는 프로그램이며, 아래 기능을 추가했다.

### 추가 기능

| 기능 | 구현 방법 |
|---|---|
| **Lo-res** | 영상을 작게 줄였다가 다시 키워 모자이크 블록 생성 |
| **Mirror** | 체크박스를 켜면 `flip`으로 좌우 반전 |

### 추가 기능 설명
**Lo-res 필터**
- 화면을 아주 작게 줄였다가 다시 원래 크기로 키우면, 픽셀이 큰 네모 블록으로 보인다.
- 키울 때 블록 모양이 그대로 보이도록 `INTER_NEAREST`를 사용했다.
- 블록 경계가 너무 각져 보여서, 작은 블러(3×3)를 줘서 부드럽게 만들었다.
- 블록 크기는 `LORES_BLOCK`, 흐림 정도는 `LORES_BLUR` 상수로 조절할 수 있다.

**Mirror 모드**
- 체크박스를 켜면 화면이 좌우 반전된다 (`cv2.flip`).
- 필터와 따로 동작해서 다른 필터와도 같이 쓸 수 있다.


In [ ]:
import cv2
import numpy as np
import tkinter as tk
AUG_TYPES = ["None", "Grayscale", "Sepia", "Sketch", "Lo-res"]
SEPIA_KERNEL = np.array([[0.272, 0.534, 0.131],
                         [0.349, 0.686, 0.168],
                         [0.393, 0.769, 0.189]])
LORES_BLOCK = 10        # 블록 크기 (클수록 더 깨짐)
LORES_BLUR = (3, 3)     # 블록 경계 흐림 정도

def apply_augmentation(frame, aug_type):
    if aug_type == "Grayscale":
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    elif aug_type == "Sepia":
        frame = cv2.transform(frame, SEPIA_KERNEL)
        frame = np.clip(frame, 0, 255)
    elif aug_type == "Sketch":
        gray_img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        inv_img = cv2.bitwise_not(gray_img)
        blur_img = cv2.GaussianBlur(inv_img, (21, 21), 0)
        inv_blur_img = cv2.bitwise_not(blur_img)
        frame = cv2.divide(gray_img, inv_blur_img, scale=256.0)
        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    elif aug_type == "Lo-res":
        h, w = frame.shape[:2]
        small = cv2.resize(frame, (w // LORES_BLOCK, h // LORES_BLOCK),
                           interpolation=cv2.INTER_AREA)            # 줄이기: 블록 평균색
        frame = cv2.resize(small, (w, h),
                           interpolation=cv2.INTER_NEAREST)         # 키우기: 각진 블록 그대로
        frame = cv2.GaussianBlur(frame, LORES_BLUR, 0)
    return frame

def adjust_brightness(frame, brightness):
    return cv2.convertScaleAbs(frame, alpha=brightness / 50)

def update():
    ret, frame = cap.read()
    if ret:
        if mirror_mode.get():
            frame = cv2.flip(frame, 1)
        frame = apply_augmentation(frame, aug_type.get())
        frame = adjust_brightness(frame, brightness_slider.get())

        img = cv2.resize(frame, (640, 480))
        png_bytes = cv2.imencode(".png", img)[1].tobytes()
        tkimg = tk.PhotoImage(data=png_bytes)
        video_label.imgtk = tkimg
        video_label.configure(image=tkimg)

    window.after(33, update)

def quit_app():
    cap.release()
    window.destroy()

if __name__ == "__main__":
    window = tk.Tk()
    window.title("Live Video Augmentation")
    cap = cv2.VideoCapture(0)
    aug_type = tk.StringVar(value="None")
    mirror_mode = tk.BooleanVar(value=False)

    video_label = tk.Label(window)
    video_label.grid(row=0, column=0, columnspan=5)
    tk.Label(window, text="Augmentation:").grid(row=1, column=0, padx=5, pady=5)
    aug_selection = tk.OptionMenu(window, aug_type, *AUG_TYPES)
    aug_selection.config(width=10)
    aug_selection.grid(row=1, column=1, padx=5, pady=5)
    brightness_slider = tk.Scale(window, from_=0, to=100, orient="horizontal", label="Brightness")
    brightness_slider.set(50)
    brightness_slider.grid(row=1, column=2, padx=5, pady=5)
    tk.Checkbutton(window, text="Mirror", variable=mirror_mode).grid(row=1, column=3, padx=5, pady=5)
    tk.Button(window, text="Quit", command=quit_app).grid(row=1, column=4, padx=5, pady=5)
    window.protocol("WM_DELETE_WINDOW", quit_app)

    update()
    window.mainloop()

2026-10-19 20:43:48.747 python[22952:140731] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
2026-10-19 20:43:49.866 python[22952:140731] +[IMKClient subclass]: chose IMKClient_Modern
2026-10-19 20:43:49.866 python[22952:140731] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 

### 필터 적용 예시

![결과 스크린샷](img/video_result.png)